# Exploratory Data Analysis – Pima Indians Diabetes Dataset

**Goal:** Understand the dataset used in the self-healing ML pipeline.

| File | Rows | Purpose |
|------|------|---------|
| `data/historical_data.csv` | 614 | Model training & baseline drift profile |
| `data/current_data.csv` | 154 | Simulated production data (used for drift detection) |

**Target column:** `Outcome` — 0 = No diabetes, 1 = Diabetes

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = os.path.join("..", "data")
hist_df = pd.read_csv(os.path.join(DATA_DIR, "historical_data.csv"))
curr_df = pd.read_csv(os.path.join(DATA_DIR, "current_data.csv"))

print("Historical shape :", hist_df.shape)
print("Current shape    :", curr_df.shape)

## 1. Schema & Basic Statistics

In [ ]:
hist_df.info()

In [ ]:
hist_df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

## 2. Missing Value Check

In [ ]:
print("Missing values in historical_data.csv:")
print(hist_df.isnull().sum())
print("\nMissing values in current_data.csv:")
print(curr_df.isnull().sum())

No missing values – all were imputed with column medians during preparation (see `scripts/prepare_data.py`).

## 3. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, df, title in zip(axes, [hist_df, curr_df], ["Historical", "Current"]):
    counts = df["Outcome"].value_counts().sort_index()
    ax.bar(["No Diabetes (0)", "Diabetes (1)"], counts, color=["steelblue", "coral"])
    ax.set_title(f"{title} – Outcome Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts):
        ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print("Historical class balance (%):")
print((hist_df["Outcome"].value_counts(normalize=True) * 100).round(1))

## 4. Feature Distributions

In [ ]:
features = [c for c in hist_df.columns if c != "Outcome"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    hist_df[hist_df["Outcome"] == 0][col].plot.hist(
        ax=ax, bins=25, alpha=0.6, color="steelblue", label="No Diabetes"
    )
    hist_df[hist_df["Outcome"] == 1][col].plot.hist(
        ax=ax, bins=25, alpha=0.6, color="coral", label="Diabetes"
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions by Class – Historical Data", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = hist_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Pearson Correlation – Historical Data", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Boxplots – Feature vs. Outcome

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    hist_df.boxplot(
        column=col,
        by="Outcome",
        ax=ax,
        patch_artist=True,
        boxprops=dict(facecolor="lightblue"),
    )
    ax.set_title(col)
    ax.set_xlabel("Outcome")

plt.suptitle("Feature vs. Outcome Boxplots – Historical Data", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Distribution Drift – Historical vs. Current

The `current_data.csv` has a subtle distribution shift on **Glucose** and **BMI** applied during data preparation to simulate real-world drift.

In [ ]:
drift_cols = ["Glucose", "BMI"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, drift_cols):
    hist_df[col].plot.hist(
        ax=ax, bins=30, alpha=0.6, color="steelblue", label="Historical", density=True
    )
    curr_df[col].plot.hist(
        ax=ax, bins=30, alpha=0.6, color="coral", label="Current", density=True
    )
    ax.set_title(f"{col} – Historical vs Current")
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Distribution Shift (Simulated Drift)", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Key Observations

| Observation | Detail |
|---|---|
| **Dataset** | Pima Indians Diabetes (768 original rows, 9 columns) |
| **Task** | Binary classification – predict diabetes onset |
| **Class balance** | ~65 % negative / 35 % positive (mild imbalance) |
| **Missing values** | Physiological zeros treated as NaN; imputed with column median |
| **Key predictors** | Glucose, BMI, and Age show the strongest correlation with Outcome |
| **Drift signal** | Glucose and BMI shifted ~3–5 % in current_data to simulate production drift |
| **Splits** | 80 % → historical_data.csv (training) / 20 % → current_data.csv (drift testing) |